In [1]:
import langchain
import json
import os
import requests
from dotenv import load_dotenv
from openai import AsyncOpenAI, OpenAI
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel
load_dotenv(override=True)

True

In [2]:
student_description = "Kennedy Mwangi is a student of computer science at University of Nairobi. He is Kenyan and has first class division. Mwangi is known for his programming skills and is an active member of university's AI club. He hopes to pursue a career in artificial intelligence after graduating."

In [3]:
student_description

"Kennedy Mwangi is a student of computer science at University of Nairobi. He is Kenyan and has first class division. Mwangi is known for his programming skills and is an active member of university's AI club. He hopes to pursue a career in artificial intelligence after graduating."

In [4]:
prompt = f'''
Please extract the following information from the given text and return it as a JSON object.

name
college
grades
club

This is the body of text to extract information from:
{student_description}
'''


In [5]:
prompt

"\nPlease extract the following information from the given text and return it as a JSON object.\n\nname\ncollege\ngrades\nclub\n\nThis is the body of text to extract information from:\nKennedy Mwangi is a student of computer science at University of Nairobi. He is Kenyan and has first class division. Mwangi is known for his programming skills and is an active member of university's AI club. He hopes to pursue a career in artificial intelligence after graduating.\n"

In [5]:
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
deepseek = OpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)

In [9]:
messages=[{
    "role": "user",
    "content": prompt
}]

In [10]:
response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages)

In [11]:
output = response.choices[0].message.content

In [12]:
output

'{\n  "name": "Kennedy Mwangi",\n  "college": "University of Nairobi",\n  "grades": "first class division",\n  "club": "AI club"\n}'

In [8]:
student_custom_function = [{
    'name': 'extract_student_info',
    'description': 'get the student information from the body of the input text',
    'parameters': {
        'type': 'object',
        'properties': {
            'name': {
                'type': 'string',
                'description': 'Name of a person'
            },
            'university': {
                'type': 'string',
                'description': 'The university name'
            },
            'grades': {
                'type': 'string',
                'description': 'Grade of the student'
            },
            'club': {
                'type': 'string',
                'description': 'Club of the student'
            }
        }
    }
}]

In [13]:
response2 = deepseek.chat.completions.create(
    model='deepseek-chat',
    messages=messages,
    functions=student_custom_function
)

In [16]:
output2 = response2.choices[0].message.content

In [19]:
output2

'```json\n{\n  "name": "Kennedy Mwangi",\n  "college": "University of Nairobi",\n  "grades": "first class division",\n  "club": "AI club"\n}\n```'

In [20]:
function_description = [{
    "name": "get_flight_info",
    "description": "get flight information between two locations",
    "parameter": {
        "type": "object",
        "properties": {
            "loc_origin": {
                "type": "string",
                "description": "The departure airport e.g. DEL"
            },
            "loc_destination": {
                "type": "string",
                "description": "The destination airport e.g. MUM"
            }
        },
        "required": ["loc_origin", "loc_destination"]
    }
}]

In [21]:
user_prompt = "When is the next flight from new delhi to mumbai?"

In [ ]:
response = deepseek.chat.completions.create(
    model="deepseek-chat",
    messages=[{
        "role": "user",
        "content": user_prompt
    }],
    functions=function_description,
    function_call="auto"
)